# IPL 2026 RAG

**Domain:** Indian Premier League 2026


## Setup

In [1]:
#install dependencies
!pip install -q \
    'Wikipedia-API>=0.14.1' \
    groq \
    chromadb \
    sentence-transformers \
    rank-bm25 \
    ragas \
    pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170

In [2]:
import shutil, os
shutil.rmtree('/content/drive', ignore_errors=True)
os.makedirs('/content/drive', exist_ok=True)

# Mount Drive and create the project folder structure on the Drive
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')


import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ipl_rag')
SUBDIRS = [
    'data/raw/wikipedia',    # JSON dumps of Wikipedia articles
    'data/raw/iplt20',       # scraped scorecards / points table
    'data/raw/espn',         # paraphrased match summaries
    'data/processed',        # cleaned, chunked text ready for embedding
    'cache',                 # LLM response cache (pickled dict)
    'embeddings',            # ChromaDB / FAISS persistence
    'results',               # eval CSVs, per-query outputs
    'reports',               # report drafts, figures
]
for sub in SUBDIRS:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_DIR}')
for sub in SUBDIRS:
    print(f'  {sub}')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
Project root: /content/drive/MyDrive/ipl_rag
  data/raw/wikipedia
  data/raw/iplt20
  data/raw/espn
  data/processed
  cache
  embeddings
  results
  reports


## API Key Configuration




In [3]:
#Paste Groq Key when prompted
from getpass import getpass
import os

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass('Groq API key: ')
print('Groq key set.' if os.environ.get('GROQ_API_KEY') else 'No key — Please add groq key.')

Groq API key: ··········
Groq key set.


## Wikipedia loader

Uses the `wikipedia-api` library and gives clean section structure. Each article is saved as JSON with
title, url, summary, full text, per-section text and categories.


In [4]:
import time

_LAST_REQUEST_TS = 0.0
MIN_REQUEST_INTERVAL = 1.0  # seconds between Wikipedia API calls

def _wiki_get(params: dict, max_retries: int = 5) -> dict:
    #GET request against the Wikipedia API with 429 handling.
    global _LAST_REQUEST_TS

    for attempt in range(max_retries):
        #MIN_REQUEST_INTERVAL between two requests
        elapsed = time.time() - _LAST_REQUEST_TS
        if elapsed < MIN_REQUEST_INTERVAL:
            time.sleep(MIN_REQUEST_INTERVAL - elapsed)

        r = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=30)
        _LAST_REQUEST_TS = time.time()

        if r.status_code == 429:
            # Honour Retry-After if Wikipedia sent one, otherwise exponential backoff
            wait = int(r.headers.get('Retry-After', 2 ** (attempt + 2)))
            print(f'429 from Wikipedia; waiting {wait}s (attempt {attempt + 1}/{max_retries})')
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r.json()

    raise RuntimeError(f'Wikipedia API still rate-limiting after {max_retries} retries')

In [5]:
import requests
import json
import re
from pathlib import Path

WIKI_DIR = PROJECT_DIR / 'data/raw/wikipedia'
WIKI_API = 'https://en.wikipedia.org/w/api.php'
WIKI_HEADERS = {
    'User-Agent': 'IPL2026_RAG_Coursework/1.0 (Warwick MSc academic project)'
}

#Fetch a Wikipedia article via the official API. Saves JSON, returns dict.
#Handles redirects: if the requested title redirects elsewhere, the file is saved under the title. If a file for that title already exists, the fetch is skipped and the existing file is returned.


def fetch_article(title: str, overwrite: bool = False) -> dict | None:
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'extracts|categories|info',
        'explaintext': 1,
        'inprop': 'url',
        'redirects': 1,
    }

    payload = _wiki_get(params)['query']

    pages = payload['pages']
    page = next(iter(pages.values()))

    if 'missing' in page or 'extract' not in page:
        print(f'  ✗ {title}: NOT FOUND')
        return None

    resolved_title = page['title']
    redirected = resolved_title != title

    #if the resolved title doesn't share meaningful words with the requested title, treat as NOT FOUND.
    def _titles_overlap(req: str, res: str) -> bool:
        req_words = set(req.lower().split())
        res_words = set(res.lower().split())
        return len(req_words & res_words) >= max(2, len(req_words) // 2)

    if redirected and not _titles_overlap(title, resolved_title):
        print(f'  ✗ {title}: NOT FOUND (got unrelated "{resolved_title}")')
        return None

    if redirected:
        print(f'  ↪ "{title}" redirected to "{resolved_title}"')


    #making sure two different requests for the same underlying article do not produce two files to avoid duplication of chunks
    safe_name = resolved_title.replace('/', '_').replace(' ', '_')
    out_path = WIKI_DIR / f'{safe_name}.json'

    if out_path.exists() and not overwrite:
        print(f'  · {resolved_title}: already on disk, skipping')
        return json.loads(out_path.read_text())

    text = page['extract']

    # Section parsing
    pattern = re.compile(r'^(={2,6})\s*(.+?)\s*\1\s*$', re.MULTILINE)
    matches = list(pattern.finditer(text))

    sections = []
    if matches:
        lead = text[:matches[0].start()].strip()
        if lead:
            sections.append({'title': 'Lead', 'level': 0, 'text': lead})
        for i, m in enumerate(matches):
            level = len(m.group(1)) - 2
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            sections.append({
                'title': m.group(2),
                'level': level,
                'text': text[start:end].strip(),
            })
    else:
        sections.append({'title': 'Lead', 'level': 0, 'text': text})

    lead_text = sections[0]['text'] if sections else ''
    summary = lead_text[:1000] + ('...' if len(lead_text) > 1000 else '')

    doc = {
        'title': resolved_title,
        'requested_title': title,  # keep for traceability if there was a redirect
        'url': page.get('fullurl', f'https://en.wikipedia.org/wiki/{resolved_title.replace(" ", "_")}'),
        'summary': summary,
        'text': text,
        'sections': sections,
        'categories': [c['title'] for c in page.get('categories', [])],
    }

    out_path.write_text(json.dumps(doc, ensure_ascii=False, indent=2))
    print(f'  ✓ {resolved_title}: {len(text):,} chars, {len(sections)} sections')
    return doc


def fetch_many(titles, overwrite=False):
    docs = {}
    for t in titles:
        d = fetch_article(t, overwrite=overwrite)
        if d is not None:
            docs[t] = d
    return docs

In [6]:
# Inspecting the file to check if we got everything alright
doc = fetch_article('2026 Indian Premier League')

if doc:
    print(f'\nTitle:    {doc["title"]}')
    print(f'URL:      {doc["url"]}')
    print(f'Length:   {len(doc["text"]):,} chars')
    print(f'Sections: {len(doc["sections"])}')
    print(f'\nFirst 10 section titles:')
    for s in doc['sections'][:10]:
        indent = '  ' * s['level']
        print(f'  {indent} {s["title"]}')
    print(f'\nSummary:')
    print(doc['summary'][:600] + '...')

  · 2026 Indian Premier League: already on disk, skipping

Title:    2026 Indian Premier League
URL:      https://en.wikipedia.org/wiki/2026_Indian_Premier_League
Length:   8,444 chars
Sections: 25

First 10 section titles:
   Lead
   Background
     Format
     Schedule
     Exclusion of Mustafizur Rahman
     Marketing
     Broadcasting
   Teams
     Personnel changes
   Venues

Summary:
The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, was the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament featured 10 teams competing in 74 matches which continued from 28 March to 31 May across 13 venues.
Defending champions Royal Challengers Bengaluru defeated Gujarat Titans by 5 wickets in the final to secure their second consecutive IPL title....


In [7]:
# Tier 1: structural articles. Titles confirmed to exist manually.
TIER_1_TITLES = [
    '2026 Indian Premier League',
    '2026 Chennai Super Kings season',
    '2026 Royal Challengers Bengaluru season',
    '2026 Mumbai Indians season',
    '2026 Gujarat Titans season',
    '2026 Delhi Capitals season',
    '2026 Sunrisers Hyderabad season',
    '2026 Lucknow Super Giants season',
    '2026 Rajasthan Royals season',
    '2026 Punjab Kings season',
    '2026 Kolkata Knight Riders season',
]

#Bulk fetching articles
docs = fetch_many(TIER_1_TITLES)
print(f'\nFetched {len(docs)} articles.')

  · 2026 Indian Premier League: already on disk, skipping
  · 2026 Chennai Super Kings season: already on disk, skipping
  · 2026 Royal Challengers Bengaluru season: already on disk, skipping
  · 2026 Mumbai Indians season: already on disk, skipping
  ✗ 2026 Gujarat Titans season: NOT FOUND (got unrelated "Indian Premier League")
  · 2026 Delhi Capitals season: already on disk, skipping
  · 2026 Sunrisers Hyderabad season: already on disk, skipping
  · 2026 Lucknow Super Giants season: already on disk, skipping
  · 2026 Rajasthan Royals season: already on disk, skipping
429 from Wikipedia; waiting 30s (attempt 1/5)
  · 2026 Punjab Kings season: already on disk, skipping
  · 2026 Kolkata Knight Riders season: already on disk, skipping

Fetched 10 articles.


## Cricbuzz scorecard scraper

Cricbuzz serves match scorecards as server-rendered HTML. Stage 1 fetches the series page to enumerate all match URLs, validates that the count is sensible, and confirms the URL structure before any per-match scraping.

In [8]:
import re
import json

import requests

CRICBUZZ_DIR = PROJECT_DIR / 'data/raw/cricbuzz'
CRICBUZZ_DIR.mkdir(parents=True, exist_ok=True)

CRICBUZZ_MATCHES_URL = (
    'https://www.cricbuzz.com/cricket-series/9241/indian-premier-league-2026/matches'
)

_HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'en-GB,en;q=0.9',
}

resp = requests.get(CRICBUZZ_MATCHES_URL, headers=_HEADERS, timeout=30)
resp.raise_for_status()
html = resp.text
print(f'Fetched {len(html):,} chars from Cricbuzz')

def _extract_match_data(html: str, target_series_id: int = 9241) -> list[dict]:
    # Find every self.__next_f.push payload string. These are strings that contain the actual serialized React tree.
    payload_pattern = re.compile(
        r'self\.__next_f\.push\(\[1,\s*"((?:[^"\\]|\\.)*)"\]\)',
        re.DOTALL,
    )
    payloads = payload_pattern.findall(html)
    print(f'Found {len(payloads)} Next.js payload chunks')

    # Join all payloads: Next.js streams them in chunks and the data we want may span boundaries.
    # JSON-decode each as a string to unescape it, then concatenate.
    combined = []
    for p in payloads:
        try:
            unescaped = json.loads('"' + p + '"')
            combined.append(unescaped)
        except json.JSONDecodeError:
            combined.append(p)
    full_text = ''.join(combined)
    print(f'Combined unescaped text: {len(full_text):,} chars')

    # match objects appear with unescaped delimiters. Walk every "matchInfo":{...} occurrence with balanced-brace parsing.
    matches_by_id = {}
    for m in re.finditer(r'"matchInfo":\{', full_text):
        start = m.end() - 1
        depth = 0
        in_string = False
        escape_next = False
        end = None
        for i in range(start, min(start + 50000, len(full_text))):
            c = full_text[i]
            if escape_next:
                escape_next = False
                continue
            if c == '\\':
                escape_next = True
                continue
            if c == '"':
                in_string = not in_string
                continue
            if in_string:
                continue
            if c == '{':
                depth += 1
            elif c == '}':
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
        if end is None:
            continue
        try:
            obj = json.loads(full_text[start:end])
        except json.JSONDecodeError:
            continue
        if obj.get('seriesId') != target_series_id:
            continue
        mid = obj.get('matchId')
        if mid is None:
            continue

        # Also try to find the sibling matchScore (lives in the parent match dict). Look just after the closing brace for ',"matchScore":{...}'
        tail = full_text[end:end + 1500]
        score_match = re.match(r',"matchScore":\{', tail)
        score_obj = None
        if score_match:
            s_start = end + score_match.end() - 1
            depth = 0
            in_string = False
            escape_next = False
            s_end = None
            for i in range(s_start, min(s_start + 5000, len(full_text))):
                c = full_text[i]
                if escape_next:
                    escape_next = False
                    continue
                if c == '\\':
                    escape_next = True
                    continue
                if c == '"':
                    in_string = not in_string
                    continue
                if in_string:
                    continue
                if c == '{':
                    depth += 1
                elif c == '}':
                    depth -= 1
                    if depth == 0:
                        s_end = i + 1
                        break
            if s_end is not None:
                try:
                    score_obj = json.loads(full_text[s_start:s_end])
                except json.JSONDecodeError:
                    pass

        # Prefer the richer version if we've seen this match before as later occurrences sometimes have more fields
        if mid not in matches_by_id or len(json.dumps(obj)) > len(json.dumps(matches_by_id[mid])):
            matches_by_id[mid] = obj
        if score_obj is not None:
            matches_by_id[mid]['matchScore'] = score_obj

    return sorted(
        matches_by_id.values(),
        key=lambda x: int(x.get('startDate', 0)),
    )


# Re-extract using the corrected parser
matches = _extract_match_data(html)

print(f'\nExtracted {len(matches)} unique IPL 2026 matches.\n')

# Save match info in json file
raw_path = CRICBUZZ_DIR / 'matches_raw.json'
raw_path.write_text(json.dumps(matches, ensure_ascii=False, indent=2))
print(f'Saved to {raw_path.name}\n')

# Quick verification
complete = [m for m in matches if m.get('state') == 'Complete']
other = [m for m in matches if m.get('state') != 'Complete']
with_score = [m for m in matches if 'matchScore' in m]
print(f'State=Complete:    {len(complete)}') #ideally should be 74 matches
print(f'State=Other:       {len(other)}')
print(f'With matchScore:   {len(with_score)}') #scores for all matches should be available

print('\nFirst 3 match info:')
for m in matches[:3]:
    t1 = m['team1']['teamSName']
    t2 = m['team2']['teamSName']
    score = m.get('matchScore', {})
    if score:
        s1 = score.get('team1Score', {}).get('inngs1', {})
        s2 = score.get('team2Score', {}).get('inngs1', {})
        score_str = f"{s1.get('runs','?')}/{s1.get('wickets','?')} vs {s2.get('runs','?')}/{s2.get('wickets','?')}"
    else:
        score_str = '(no score)'
    print(f"  [{m['matchId']}] {m['matchDesc']}: {t1} vs {t2} — {m.get('status','TBC')} — {score_str}")

print('\nLast 3 match info:')
for m in matches[-3:]:
    t1 = m['team1']['teamSName']
    t2 = m['team2']['teamSName']
    score = m.get('matchScore', {})
    if score:
        s1 = score.get('team1Score', {}).get('inngs1', {})
        s2 = score.get('team2Score', {}).get('inngs1', {})
        score_str = f"{s1.get('runs','?')}/{s1.get('wickets','?')} vs {s2.get('runs','?')}/{s2.get('wickets','?')}"
    else:
        score_str = '(no score)'
    print(f"  [{m['matchId']}] {m['matchDesc']}: {t1} vs {t2} — {m.get('status','TBC')} — {score_str}")

Fetched 366,677 chars from Cricbuzz
Found 44 Next.js payload chunks
Combined unescaped text: 275,975 chars

Extracted 74 unique IPL 2026 matches.

Saved to matches_raw.json

State=Complete:    0
State=Other:       74
With matchScore:   74

First 3 match info:
  [149618] 1st Match: RCB vs SRH — Royal Challengers Bengaluru won by 6 wkts — 203/4 vs 201/9
  [149629] 2nd Match: MI vs KKR — Mumbai Indians won by 6 wkts — 224/4 vs 220/4
  [149640] 3rd Match: RR vs CSK — Rajasthan Royals won by 8 wkts — 128/2 vs 127/10

Last 3 match info:
  [155387] Eliminator: SRH vs RR — Rajasthan Royals won by 47 runs — 196/10 vs 243/8
  [155398] Qualifier 2: GT vs RR — Gujarat Titans won by 7 wkts — 219/3 vs 214/6
  [155409] Final: RCB vs GT — Royal Challengers Bengaluru won by 5 wkts — 161/5 vs 155/8


## Match data to prose chunks

Render each match dict as a self-contained prose paragraph. Each match becomes
one chunk. Metadata fields mirror the Wikipedia chunks.

In [9]:
from transformers import AutoTokenizer
TOKENIZER = AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [10]:
import json
from datetime import datetime, timezone
from pathlib import Path

CRICBUZZ_CHUNKS_PATH = PROJECT_DIR / 'data/processed/cricbuzz_chunks.json'


def _format_inning_score(team_name: str, innings: dict | None) -> str:
    # render '{Team} scored 201/9 in 20 overs' style sentence fragment
    if not innings:
        return f'{team_name} did not bat'
    runs = innings.get('runs', '?')
    wickets = innings.get('wickets', '?')
    overs = innings.get('overs', '?')
    return f'{team_name} scored {runs}/{wickets} in {overs} overs'


def _format_match_date(start_date_ms: str | int) -> str:
    # render the millisecond timestamp as 'DD Month YYYY'
    try:
        ts = int(start_date_ms) / 1000
        dt = datetime.fromtimestamp(ts, tz=timezone.utc)
        return dt.strftime('%d %B %Y')
    except (TypeError, ValueError):
        return 'unknown date'


def match_to_prose(match: dict) -> str | None:
    #Turn one match dict into a prose chunk. Returns None if no score available
    score = match.get('matchScore')
    if not score:
        return None

    desc = match.get('matchDesc', 'a match')
    t1 = match['team1']['teamName']
    t2 = match['team2']['teamName']
    t1_short = match['team1']['teamSName']
    t2_short = match['team2']['teamSName']

    venue = match.get('venueInfo', {})
    ground = venue.get('ground', 'an unknown venue')
    city = venue.get('city', '')
    venue_str = f'{ground}, {city}' if city else ground

    date_str = _format_match_date(match.get('startDate', ''))
    status = match.get('status', '').strip()

    t1_innings = score.get('team1Score', {}).get('inngs1')
    t2_innings = score.get('team2Score', {}).get('inngs1')

    lines = [
        f'In the {desc} of the 2026 Indian Premier League (IPL 2026), '
        f'played at {venue_str} on {date_str}, {t1} ({t1_short}) faced '
        f'{t2} ({t2_short}).',
        f'{_format_inning_score(t1, t1_innings)}.',
        f'{_format_inning_score(t2, t2_innings)}.',
    ]

    if status:
        lines.append(f'Result: {status}.')

    # Handle the super-over case (2nd innings)
    t1_super = score.get('team1Score', {}).get('inngs2')
    t2_super = score.get('team2Score', {}).get('inngs2')
    if t1_super or t2_super:
        lines.append('A super over followed:')
        if t1_super:
            lines.append(f'  {_format_inning_score(t1, t1_super)} in the super over.')
        if t2_super:
            lines.append(f'  {_format_inning_score(t2, t2_super)} in the super over.')

    return ' '.join(lines)


def match_to_chunk(match: dict) -> dict | None:
    #Turn one match dict into a chunk record matching the Wikipedia chunk schema
    text = match_to_prose(match)
    if text is None:
        return None

    mid = match['matchId']
    desc = match.get('matchDesc', '')
    t1_short = match['team1']['teamSName']
    t2_short = match['team2']['teamSName']

    venue = match.get('venueInfo', {})

    # Schema mirrors the Wikipedia chunk schema: id, text, metadata
    return {
        'chunk_id': f'cricbuzz_match::{mid}',
        'text': text,
        'metadata': {
            'source_title': f'IPL 2026 — {desc}: {t1_short} vs {t2_short}',
            'source_url': f'https://www.cricbuzz.com/live-cricket-scorecard/{mid}',
            'section_title': desc or 'Match',
            'section_level': 0,
            'section_index': 0,
            'piece_index': 0,
            'n_tokens': len(TOKENIZER.encode(text, add_special_tokens=False)),
            'source_type': 'cricbuzz_match',  # new field to distinguish between cricbuzz and wikipedia chunks
            'match_id': str(mid),
            'date_ms': str(match.get('startDate', '')),
        },
    }


# Load extracted matches and convert
matches = json.loads((CRICBUZZ_DIR / 'matches_raw.json').read_text())

chunks = []
skipped = []
for m in matches:
    chunk = match_to_chunk(m)
    if chunk is None:
        skipped.append(m.get('matchId'))
        continue
    chunks.append(chunk)

# Save
CRICBUZZ_CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2))
print(f'Rendered {len(chunks)} matches as prose chunks.')
if skipped:
    print(f'Skipped {len(skipped)} matches without scores yet: {skipped}')
print(f'Saved to {CRICBUZZ_CHUNKS_PATH.name}\n')

# Preview
print('First chunk preview:')
print(chunks[0]['text'])
print(f'\nMetadata: {chunks[0]["metadata"]}')

print('\n\nLast chunk preview:')
print(chunks[-1]['text'])

Rendered 74 matches as prose chunks.
Saved to cricbuzz_chunks.json

First chunk preview:
In the 1st Match of the 2026 Indian Premier League (IPL 2026), played at M.Chinnaswamy Stadium, Bengaluru on 28 March 2026, ROYAL CHALLENGERS BENGALURU (RCB) faced SUNRISERS HYDERABAD (SRH). ROYAL CHALLENGERS BENGALURU scored 203/4 in 15.4 overs. SUNRISERS HYDERABAD scored 201/9 in 20 overs. Result: Royal Challengers Bengaluru won by 6 wkts.

Metadata: {'source_title': 'IPL 2026 — 1st Match: RCB vs SRH', 'source_url': 'https://www.cricbuzz.com/live-cricket-scorecard/149618', 'section_title': '1st Match', 'section_level': 0, 'section_index': 0, 'piece_index': 0, 'n_tokens': 94, 'source_type': 'cricbuzz_match', 'match_id': '149618', 'date_ms': '1774706400000'}


Last chunk preview:
In the Final of the 2026 Indian Premier League (IPL 2026), played at Narendra Modi Stadium, Ahmedabad on 31 May 2026, ROYAL CHALLENGERS BENGALURU (RCB) faced GUJARAT TITANS (GT). ROYAL CHALLENGERS BENGALURU scored 161/5 in

## Groq client with persistent cache

**Cache key** = `sha256(query + pipeline_version + sha256(prompt))`. This means:
- Same query + same prompt + same pipeline → instant replay, zero tokens
- Change the prompt template? New key, fresh call (intentional — you want to see the diff)
- Change pipeline version (`v0_baseline` → `v1_hybrid`)? Fresh call too

The cache survives Colab restarts because it's pickled to Drive after every successful call.
Keep the file under a few hundred MB — at typical RAG response sizes you'd need tens of
thousands of queries to get close.

In [11]:
import hashlib
import json
import pickle
from pathlib import Path

CACHE_PATH = PROJECT_DIR / 'cache' / 'llm_cache.pkl'

def _load_cache() -> dict:
    if CACHE_PATH.exists():
        try:
            return pickle.loads(CACHE_PATH.read_bytes())
        except Exception as e:
            print(f'Cache load failed ({e}); starting fresh.')
            return {}
    return {}

def _save_cache(cache: dict) -> None:
    # Write to a temp file then rename, avoids a half-written pickle if Colab dies mid-write.
    tmp = CACHE_PATH.with_suffix('.pkl.tmp')
    tmp.write_bytes(pickle.dumps(cache))
    tmp.replace(CACHE_PATH)

def _cache_key(query: str, pipeline_version: str, prompt: str) -> str:
    payload = json.dumps({
        'query': query,
        'pipeline': pipeline_version,
        'prompt_hash': hashlib.sha256(prompt.encode('utf-8')).hexdigest(),
    }, sort_keys=True)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

In [12]:
from groq import Groq
import time

_client = Groq()  # read GROQ_API_KEY from env

MODELS = {
    'fast':    'llama-3.1-8b-instant',     # prompt iteration, cheap, fast
    'quality': 'llama-3.3-70b-versatile',  # real eval runs
}

_llm_cache = _load_cache()
print(f'Loaded LLM cache: {len(_llm_cache)} entries')


def llm_complete(
    query: str,
    prompt: str,
    pipeline_version: str = 'v0_baseline',
    model: str = 'quality',
    temperature: float = 0.0,
    max_tokens: int = 800,
    use_cache: bool = True,
) -> str:
    """Call Groq with caching + exponential backoff.

    `query` is the user question (cache key component).
    `prompt` is the full prompt string sent to the model (system + context + question).
    `pipeline_version` distinguishes baseline vs enhanced for the same query.
    """
    key = _cache_key(query, pipeline_version, prompt)
    if use_cache and key in _llm_cache:
        return _llm_cache[key]['response']

    last_err = None
    for attempt in range(4):
        try:
            r = _client.chat.completions.create(
                model=MODELS[model],
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            response = r.choices[0].message.content
            _llm_cache[key] = {
                'response': response,
                'model': MODELS[model],
                'pipeline': pipeline_version,
                'query': query,
            }
            _save_cache(_llm_cache)
            return response
        except Exception as e:
            last_err = e
            wait = 2 ** attempt  # 1s, 2s, 4s, 8s
            print(f'  attempt {attempt + 1} failed ({type(e).__name__}: {e}); retrying in {wait}s')
            time.sleep(wait)

    raise RuntimeError(f'LLM call failed after 4 attempts. Last error: {last_err}')


def cache_stats() -> None:
    #Quick look at what's in the cache, grouped by pipeline
    from collections import Counter
    pipelines = Counter(v['pipeline'] for v in _llm_cache.values())
    models = Counter(v['model'] for v in _llm_cache.values())
    print(f'Total cached: {len(_llm_cache)}')
    print(f'By pipeline:  {dict(pipelines)}')
    print(f'By model:     {dict(models)}')

Loaded LLM cache: 77 entries


## End-to-end test

In [13]:
# Cheap call on the 8B model to confirm the wrapper works.
response = llm_complete(
    query='hello world',
    prompt='Reply with exactly this sentence and nothing else: "RAG is ready."',
    pipeline_version='smoke_test',
    model='fast',
)
print(response)
print()
cache_stats()

RAG is ready.

Total cached: 77
By pipeline:  {'smoke_test': 2, 'v0_baseline': 39, 'v0_baseline_with_cricbuzz': 8, 'v1_hybrid_rerank': 28}
By model:     {'llama-3.1-8b-instant': 2, 'llama-3.3-70b-versatile': 75}


In [14]:
# Re-running the same call to make sure it calls the cache and not groq
response2 = llm_complete(
    query='hello world',
    prompt='Reply with exactly this sentence and nothing else: "RAG is ready."',
    pipeline_version='smoke_test',
    model='fast',
)
print(response2)
print('Cache hit confirmed.' if response == response2 else 'WARNING: responses differ')

RAG is ready.
Cache hit confirmed.


## Chunking

Split each saved Wikipedia article into 512-token chunks with 50-token overlap.
Section boundaries are respected and a single chunk never crosses from one section
into the next. Each chunk carries metadata (source title, section, position) so retrieval results can be
attributed back to where they came from.

In [15]:
import json
from pathlib import Path
from transformers import AutoTokenizer

CHUNKS_PATH = PROJECT_DIR / 'data/processed/chunks.json'

# Use the same tokenizer the embedding model will use, keeps "token count" honest.
TOKENIZER = AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')

CHUNK_SIZE = 512      # target tokens per chunk
CHUNK_OVERLAP = 50    # tokens of overlap between adjacent chunks within a section
MIN_CHUNK_TOKENS = 30 # drop chunks tinier than this


def _split_text_by_tokens(text: str, size: int, overlap: int) -> list[str]:
    #Token-aware splitter. Decodes back to text so chunks are readable
    token_ids = TOKENIZER.encode(text, add_special_tokens=False)
    if len(token_ids) <= size:
        return [text]

    step = size - overlap
    chunks = []
    for start in range(0, len(token_ids), step):
        window = token_ids[start:start + size]
        if len(window) < MIN_CHUNK_TOKENS and chunks:
            break  # tail too small, already covered by previous chunk's overlap
        chunks.append(TOKENIZER.decode(window, skip_special_tokens=True))
        if start + size >= len(token_ids):
            break
    return chunks


def chunk_article(doc: dict) -> list[dict]:
    #Turn one article dict into a list of chunk records
    out = []
    for sec_idx, section in enumerate(doc['sections']):
        sec_text = section['text'].strip()
        if not sec_text:
            continue
        pieces = _split_text_by_tokens(sec_text, CHUNK_SIZE, CHUNK_OVERLAP)
        for piece_idx, piece in enumerate(pieces):
            out.append({
                'chunk_id': f"{doc['title']}::{sec_idx:02d}::{piece_idx:02d}",
                'text': piece,
                'metadata': {
                    'source_title': doc['title'],
                    'source_url': doc['url'],
                    'section_title': section['title'],
                    'section_level': section['level'],
                    'section_index': sec_idx,
                    'piece_index': piece_idx,
                    'n_tokens': len(TOKENIZER.encode(piece, add_special_tokens=False)),
                },
            })
    return out


def chunk_corpus(wiki_dir: Path = WIKI_DIR) -> list[dict]:
    #Load every saved Wikipedia JSON and chunk it
    all_chunks = []
    files = sorted(wiki_dir.glob('*.json'))
    for f in files:
        doc = json.loads(f.read_text())
        chunks = chunk_article(doc)
        all_chunks.extend(chunks)
    return all_chunks


# Run chunking on Wikipedia articles
wiki_chunks = chunk_corpus()

# Load the Cricbuzz match chunks built in section 13
cricbuzz_chunks_path = PROJECT_DIR / 'data/processed/cricbuzz_chunks.json'
cricbuzz_chunks = json.loads(cricbuzz_chunks_path.read_text()) if cricbuzz_chunks_path.exists() else []

# Combine into one chunk list and persist
chunks = wiki_chunks + cricbuzz_chunks
CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2))

print(f'\nWikipedia chunks: {len(wiki_chunks)}')
print(f'Cricbuzz chunks: {len(cricbuzz_chunks)}')
print(f'Total: {len(chunks)} chunks')
print(f'Saved to: {CHUNKS_PATH}')


Wikipedia chunks: 71
Cricbuzz chunks: 74
Total: 145 chunks
Saved to: /content/drive/MyDrive/ipl_rag/data/processed/chunks.json


## Embeddings and vector store

Encode every chunk with `BAAI/bge-small-en-v1.5`. Store vectors and metadata in a persistent ChromaDB collection
on Drive. Subsequent sessions reload from disk instead of re-embedding.

In [16]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

CHROMA_DIR = PROJECT_DIR / 'embeddings' / 'chroma'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
COLLECTION_NAME = 'ipl2026_v1'

EMBED_MODEL_NAME = 'BAAI/bge-small-en-v1.5'
EMBED_BATCH_SIZE = 32

# Load the embedding model
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f'Loaded {EMBED_MODEL_NAME}: dim={embed_model.get_sentence_embedding_dimension()}')


def embed_texts(texts: list[str], is_query: bool = False) -> list[list[float]]:
    """Encode a list of texts. BGE wants a query prefix for questions, not for documents."""
    if is_query:
        texts = [f'Represent this sentence for searching relevant passages: {t}' for t in texts]
    vectors = embed_model.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,  # cosine similarity becomes a dot product
        convert_to_numpy=True,
    )
    return vectors.tolist()


# Set up Chroma with on-disk persistence
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)


def build_collection(chunks: list[dict], rebuild: bool = False):
    #Create or refresh the Chroma collection from the chunk list
    existing = [c.name for c in chroma_client.list_collections()]
    if COLLECTION_NAME in existing:
        if rebuild:
            chroma_client.delete_collection(COLLECTION_NAME)
            print(f'Deleted existing collection "{COLLECTION_NAME}"')
        else:
            coll = chroma_client.get_collection(COLLECTION_NAME)
            print(f'Collection "{COLLECTION_NAME}" already exists with {coll.count()} docs')
            print('Pass rebuild=True to recreate from scratch.')
            return coll

    coll = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={'hnsw:space': 'cosine'},
    )

    # Embed and insert in batches so memory stays sensible
    BATCH = 64
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i:i + BATCH]
        ids = [c['chunk_id'] for c in batch]
        texts = [c['text'] for c in batch]
        metas = [c['metadata'] for c in batch]
        vectors = embed_texts(texts, is_query=False)
        coll.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
        print(f'  inserted {min(i + BATCH, len(chunks))}/{len(chunks)}')

    print(f'\nCollection "{COLLECTION_NAME}" built: {coll.count()} chunks')
    return coll


# Load chunks from the previous step and build the collection
chunks = json.loads(CHUNKS_PATH.read_text())
print(f'Loaded {len(chunks)} chunks from {CHUNKS_PATH.name}\n')
collection = build_collection(chunks, rebuild=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_10254/280832561.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Loaded {EMBED_MODEL_NAME}: dim={embed_model.get_sentence_embedding_dimension()}')


Loaded BAAI/bge-small-en-v1.5: dim=384
Loaded 145 chunks from chunks.json

Collection "ipl2026_v1" already exists with 145 docs
Pass rebuild=True to recreate from scratch.


## Dense retrieval (baseline model)


In [17]:
def retrieve_dense(query: str, k: int = 5, verbose: bool = False) -> list[dict]:
    #Given a question, embed it with the same model (BGE, with query prefix) and ask Chroma for the top-k most similar chunks by cosine similarity.
    #Each result is a dict with: chunk_id, text, metadata, score (higher = better).

    # Embed the query with the BGE query prefix
    query_vector = embed_texts([query], is_query=True)[0]

    # Ask Chroma for top-k. n_results is the only knob, everything else lives in the collection.
    raw = collection.query(
        query_embeddings=[query_vector],
        n_results=k,
        include=['documents', 'metadatas', 'distances'],
    )


    results = []
    for chunk_id, text, meta, dist in zip(
        raw['ids'][0],
        raw['documents'][0],
        raw['metadatas'][0],
        raw['distances'][0],
    ):
        results.append({
            'chunk_id': chunk_id,
            'text': text,
            'metadata': meta,
            'score': 1.0 - dist,
        })

    if verbose:
        print(f'Query: {query!r}')
        for i, r in enumerate(results, 1):
            preview = r['text'].replace('\n', ' ')[:140]
            print(f'  [{i}] {r["score"]:.3f}  {r["metadata"]["source_title"]} '
                  f'§ {r["metadata"]["section_title"]}')
            print(f'      {preview}...')
    return results


# Smoke test
_ = retrieve_dense('Who won the IPL 2026 final?', k=5, verbose=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'Who won the IPL 2026 final?'
  [1] 0.829  2026 Indian Premier League § Lead
      The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, was the 19th edition of the Indian Premier League, a prof...
  [2] 0.827  2026 Sunrisers Hyderabad season § Pre-season
      The 2026 Indian Premier League was the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cricket league, organi...
  [3] 0.819  2026 Mumbai Indians season § Pre-season
      The 2026 Indian Premier League is the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cricket league, organis...
  [4] 0.818  2026 Chennai Super Kings season § Pre-season
      The 2026 Indian Premier League will be the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cricket league hel...
  [5] 0.814  2026 Mumbai Indians season § Lead
      The 2026 season was the 19th season for the Indian Premier League (IPL) cricket franchise Mumb

## Grounded prompt and generation

Wires retrieval to the LLM. The prompt template tells the model to answer only
from the retrieved passages, to cite which passage each claim comes from, and
to refuse explicitly when the passages don't contain the answer.

`rag_answer(query)` is the full baseline pipeline with retrieve, format, generate in one function. This is what the evaluation phase will call.

In [18]:
GROUNDED_PROMPT_TEMPLATE = """You are a cricket analyst answering questions about the 2026 Indian Premier League (IPL 2026).

Answer the question using ONLY the numbered passages provided below. Follow these rules strictly:

1. Every factual claim in your answer must be supported by at least one passage. Do not use any outside knowledge, including general knowledge about cricket or previous IPL seasons.

2. After each claim, cite the supporting passage number(s) in square brackets, e.g. "Royal Challengers Bengaluru finished second [2]." or "CSK won both league fixtures against MI [1][3]."

3. If the passages do not contain enough information to answer the question, reply with exactly: "The provided context does not answer this question." Do not guess, infer beyond the passages, or fall back on general knowledge.

4. If the passages contain conflicting information, note the conflict rather than picking one side silently.

5. Be concise. Do not pad the answer with restatements of the question or filler.

---

PASSAGES:
{context}

---

QUESTION: {question}

ANSWER:"""


def format_passages(results: list[dict]) -> str:
    #Render retrieval results as a numbered passage block for the prompt
    blocks = []
    for i, r in enumerate(results, 1):
        meta = r['metadata']
        header = f"[{i}] Source: {meta['source_title']} — {meta['section_title']}"
        blocks.append(f"{header}\n{r['text']}")
    return '\n\n'.join(blocks)


def rag_answer(
    query: str,
    k: int = 5,
    model: str = 'quality',
    pipeline_version: str = 'v0_baseline',
    verbose: bool = False,
) -> dict:
    """End-to-end baseline RAG: retrieve, prompt, generate.
    Returns a dict with the answer, the retrieved chunks, and the full prompt
    sent to the LLM.
    """
    results = retrieve_dense(query, k=k, verbose=verbose)
    context = format_passages(results)
    prompt = GROUNDED_PROMPT_TEMPLATE.format(context=context, question=query)

    answer = llm_complete(
        query=query,
        prompt=prompt,
        pipeline_version=pipeline_version,
        model=model,
    )

    return {
        'query': query,
        'answer': answer,
        'retrieved': results,
        'prompt': prompt,
        'pipeline_version': pipeline_version,
    }


# Smoke test
out = rag_answer('Who won the IPL 2026 final?', verbose=True)
print('ANSWER:')
print(out['answer'])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'Who won the IPL 2026 final?'
  [1] 0.829  2026 Indian Premier League § Lead
      The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, was the 19th edition of the Indian Premier League, a prof...
  [2] 0.827  2026 Sunrisers Hyderabad season § Pre-season
      The 2026 Indian Premier League was the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cricket league, organi...
  [3] 0.819  2026 Mumbai Indians season § Pre-season
      The 2026 Indian Premier League is the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cricket league, organis...
  [4] 0.818  2026 Chennai Super Kings season § Pre-season
      The 2026 Indian Premier League will be the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cricket league hel...
  [5] 0.814  2026 Mumbai Indians season § Lead
      The 2026 season was the 19th season for the Indian Premier League (IPL) cricket franchise Mumb

## Test queries

Sanity check wiht ten questions spanning the three question types the report will frame around:
fact retrieval, cross-document reasoning, and narrative/contextual. Plus a couple of deliberate out-of-corpus questions to confirm the refusal clause works.


In [20]:
SMOKE_QUERIES = [
    # Fact retrieval
    'Which team won the IPL 2026 title?',
    'Who was the captain of Chennai Super Kings in IPL 2026?',
    'When did the IPL 2026 season begin?',
    'How many teams competed in IPL 2026?',

    # Cross-document / multi-chunk reasoning
    'How did Royal Challengers Bengaluru perform in IPL 2026 compared to their 2025 season?',
    'Which teams qualified for the playoffs in IPL 2026?',

    # Narrative / contextual
    'What were the major storylines of the IPL 2026 season?',
    'How did the Chennai Super Kings season unfold in IPL 2026?',

    # Out-of-corpus (should trigger the refusal clause)
    'What was the attendance at the IPL 2026 final?',
    'Who won the Orange Cap in IPL 2024?',
]


def run_smoke_tests(queries: list[str], k: int = 5) -> list[dict]:
    #Runs each query through the baseline pipeline and print a compact trace
    runs = []
    for i, q in enumerate(queries, 1):
        print(f'[{i}/{len(queries)}] {q}')
        print('=' * 70)

        out = rag_answer(q, k=k, verbose=False)

        # Compact retrieval trace: one line per chunk
        print('Retrieved:')
        for j, r in enumerate(out['retrieved'], 1):
            print(f'  [{j}] {r["score"]:.3f}  {r["metadata"]["source_title"]} '
                  f'§ {r["metadata"]["section_title"]}')

        print(f'\nAnswer:\n{out["answer"]}')
        runs.append(out)
    return runs


smoke_results = run_smoke_tests(SMOKE_QUERIES)


# Save the run to refer back to it and be able to contrast with enhanced runs later
import json
from datetime import datetime

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
out_path = PROJECT_DIR / 'results' / f'smoke_baseline_{stamp}.json'
serialisable = [
    {
        'query': r['query'],
        'answer': r['answer'],
        'pipeline_version': r['pipeline_version'],
        'retrieved': [
            {'chunk_id': c['chunk_id'], 'score': c['score'], 'metadata': c['metadata']}
            for c in r['retrieved']
        ],
    }
    for r in smoke_results
]
out_path.write_text(json.dumps(serialisable, ensure_ascii=False, indent=2))
print(f'\nSaved smoke run to {out_path.name}')

[1/10] Which team won the IPL 2026 title?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.854  2026 Sunrisers Hyderabad season § Pre-season
  [2] 0.841  2026 Mumbai Indians season § Pre-season
  [3] 0.838  2026 Chennai Super Kings season § Pre-season
  [4] 0.838  2026 Indian Premier League § Lead
  [5] 0.813  2026 Mumbai Indians season § Lead

Answer:
Royal Challengers Bengaluru won the IPL 2026 title [4].
[2/10] Who was the captain of Chennai Super Kings in IPL 2026?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.865  2026 Chennai Super Kings season § Lead
  [2] 0.801  2026 Chennai Super Kings season § Pre-season
  [3] 0.799  2026 Sunrisers Hyderabad season § Lead
  [4] 0.782  IPL 2026 — 22nd Match: CSK vs KKR § 22nd Match
  [5] 0.779  IPL 2026 — 18th Match: CSK vs DC § 18th Match

Answer:
The captain of Chennai Super Kings in IPL 2026 was Ruturaj Gaikwad [1].
[3/10] When did the IPL 2026 season begin?


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.814  2026 Indian Premier League § Lead
  [2] 0.808  2026 Chennai Super Kings season § Pre-season
  [3] 0.803  2026 Mumbai Indians season § Pre-season
  [4] 0.788  2026 Sunrisers Hyderabad season § Pre-season
  [5] 0.776  IPL 2026 — 20th Match: RCB vs MI § 20th Match

Answer:
The provided context does not answer this question.
[10/10] Who won the Orange Cap in IPL 2024?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.760  2026 Sunrisers Hyderabad season § Pre-season
  [2] 0.739  2026 Mumbai Indians season § Pre-season
  [3] 0.725  2026 Indian Premier League § Lead
  [4] 0.723  2026 Mumbai Indians season § Lead
  [5] 0.719  2026 Chennai Super Kings season § Pre-season

Answer:
The provided context does not answer this question.

Saved smoke run to smoke_baseline_20260605_130713.json


/tmp/ipykernel_10254/2679280386.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')


In [21]:
# Questions that should now be answerable with the expanded corpus
SMOKE_QUERIES_V2 = [
    # GT / LSG / PBKS / RR — teams that had no Wikipedia coverage
    'How did Gujarat Titans perform in IPL 2026?',
    'What was the result of the Lucknow Super Giants opening match?',
    'Did Punjab Kings beat Rajasthan Royals in IPL 2026?',

    # Per-match factual questions — previously hard, now should work
    'Who won the 1st match of IPL 2026?',
    'What was the score in the CSK vs MI match at Wankhede?',
    'Were there any super-over matches in IPL 2026?',

    # Cross-document — needs both Wikipedia and Cricbuzz
    'How did Royal Challengers Bengaluru perform in IPL 2026?',
    'Which team won the most matches in the league stage of IPL 2026?',
]

# Use a new pipeline_version so we don't get cached answers from the smaller corpus
smoke_results_v2 = []
for q in SMOKE_QUERIES_V2:
    print(f'\n{"=" * 70}')
    print(f'Q: {q}')
    print('=' * 70)
    out = rag_answer(q, k=5, pipeline_version='v0_baseline_with_cricbuzz')

    print('Retrieved:')
    for j, r in enumerate(out['retrieved'], 1):
        src_type = r['metadata'].get('source_type', 'wikipedia')
        print(f"  [{j}] {r['score']:.3f}  ({src_type})  "
              f"{r['metadata']['source_title']} § {r['metadata']['section_title']}")

    print(f'\nA: {out["answer"]}')
    smoke_results_v2.append(out)

#baseline failing like when asked the winner of 1st match, it wasn't able to find the context



Q: How did Gujarat Titans perform in IPL 2026?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.866  (cricbuzz_match)  IPL 2026 — 66th Match: GT vs CSK § 66th Match
  [2] 0.855  (cricbuzz_match)  IPL 2026 — 52nd Match: GT vs RR § 52nd Match
  [3] 0.854  (cricbuzz_match)  IPL 2026 — 37th Match: CSK vs GT § 37th Match
  [4] 0.851  (cricbuzz_match)  IPL 2026 — 46th Match: PBKS vs GT § 46th Match
  [5] 0.851  (cricbuzz_match)  IPL 2026 — 42nd Match: RCB vs GT § 42nd Match

A: Gujarat Titans won against Chennai Super Kings by 89 runs [1] and by 8 wickets [3], against Rajasthan Royals by 77 runs [2], against Punjab Kings by 4 wickets [4], and against Royal Challengers Bengaluru by 4 wickets [5].

Q: What was the result of the Lucknow Super Giants opening match?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.776  (cricbuzz_match)  IPL 2026 — 59th Match: CSK vs LSG § 59th Match
  [2] 0.774  (cricbuzz_match)  IPL 2026 — 19th Match: LSG vs GT § 19th Match
  [3] 0.772  (cricbuzz_match)  IPL 2026 — 53rd Match: LSG vs CSK § 53rd Match
  [4] 0.772  (cricbuzz_match)  IPL 2026 — 5th Match: LSG vs DC § 5th Match
  [5] 0.769  (cricbuzz_match)  IPL 2026 — 68th Match: LSG vs PBKS § 68th Match

A: The provided context does not answer this question.

Q: Did Punjab Kings beat Rajasthan Royals in IPL 2026?


...

Retrieved:
  [1] 0.824  (wikipedia)  2026 Chennai Super Kings season § Pre-season
  [2] 0.816  (wikipedia)  2026 Indian Premier League § Lead
  [3] 0.801  (cricbuzz_match)  IPL 2026 — 22nd Match: CSK vs KKR § 22nd Match
  [4] 0.797  (wikipedia)  2026 Mumbai Indians season § Pre-season
  [5] 0.796  (cricbuzz_match)  IPL 2026 — 33rd Match: CSK vs MI § 33rd Match

A: The provided context does not answer this question.

Q: How did Royal Challengers Bengaluru perform in IPL 2026?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.882  (wikipedia)  2026 Royal Challengers Bengaluru season § Lead
  [2] 0.862  (cricbuzz_match)  IPL 2026 — 11th Match: RCB vs CSK § 11th Match
  [3] 0.858  (wikipedia)  2026 Royal Challengers Bengaluru season § Pre-season
  [4] 0.846  (cricbuzz_match)  IPL 2026 — 61st Match: RCB vs PBKS § 61st Match
  [5] 0.845  (cricbuzz_match)  IPL 2026 — 16th Match: RCB vs RR § 16th Match

A: Royal Challengers Bengaluru entered the 2026 Indian Premier League as the defending champions [1]. They won against Chennai Super Kings by 43 runs [2] and against Punjab Kings by 23 runs [4], but lost to Rajasthan Royals by 6 wickets [5]. [1][2][4][5]

Q: Which team won the most matches in the league stage of IPL 2026?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved:
  [1] 0.823  (wikipedia)  2026 Indian Premier League § Lead
  [2] 0.813  (wikipedia)  2026 Mumbai Indians season § Pre-season
  [3] 0.812  (wikipedia)  2026 Sunrisers Hyderabad season § Pre-season
  [4] 0.812  (wikipedia)  2026 Chennai Super Kings season § Pre-season
  [5] 0.772  (cricbuzz_match)  IPL 2026 — 63rd Match: CSK vs SRH § 63rd Match

A: The provided context does not answer this question.


## BM25 retrieval (sparse, lexical)

BM25 is a classical keyword-based retrieval algorithm. It scores documents by
how rarely a query's terms appear in the corpus and how concentrated they are
in each document. Dense retrieval (BGE embeddings) captures semantic
similarity, whereas BM25 captures exact lexical overlap, including names, scores,
match descriptors in our data like "1st Match", and other rare terms that the embedding model can dilute.

BM25 alongside dense retrieval (hybrid retreival), it catches what dense misses.

In [22]:
import json
import re
from pathlib import Path
from rank_bm25 import BM25Okapi

# Tokenize the same chunks that went into Chroma
chunks = json.loads(CHUNKS_PATH.read_text())
print(f'Loaded {len(chunks)} chunks for BM25 indexing.\n')

def _tokenize_for_bm25(text: str) -> list[str]:
    """Lowercase, strip punctuation, split on whitespace.

    Keeps numbers (so '203/4' and '20' are tokens). Keeps the slash in scores.
    Splits on / and - for safety so 'Sunrisers Hyderabad-RCB' becomes three
    tokens rather than one.
    """
    text = text.lower()
    # Keep alphanumerics and slashes (for scores like 203/4) but split on everything else
    tokens = re.findall(r"[a-z0-9/]+", text)
    return tokens


# Build the corpus once, in the same order as `chunks`
tokenized_corpus = [_tokenize_for_bm25(c['text']) for c in chunks]
bm25_index = BM25Okapi(tokenized_corpus)
print(f'BM25 index built on {len(tokenized_corpus)} chunks.')

# Preview the tokenization on one chunk so we can sanity-check it
sample_chunk = chunks[58]  # first Cricbuzz chunk in our corpus
print(f'\nSample tokenization preview:')
print(f'  Source: {sample_chunk["metadata"]["source_title"]}')
print(f'  Text: {sample_chunk["text"][:160]}...')
print(f'  Tokens (first 20): {tokenized_corpus[58][:20]}')


def retrieve_bm25(query: str, k: int = 5, verbose: bool = False) -> list[dict]:
    #Returns top-k chunks ranked by BM25 score.
    tokens = _tokenize_for_bm25(query)
    scores = bm25_index.get_scores(tokens)

    # Take top-k indices
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

    results = []
    for idx in top_indices:
        c = chunks[idx]
        results.append({
            'chunk_id': c['chunk_id'],
            'text': c['text'],
            'metadata': c['metadata'],
            'score': float(scores[idx]),
        })

    if verbose:
        print(f'\nQuery: {query!r}')
        for i, r in enumerate(results, 1):
            src = r['metadata'].get('source_type', 'wikipedia')
            preview = r['text'].replace('\n', ' ')[:140]
            print(f'  [{i}] {r["score"]:6.2f}  ({src:15s})  '
                  f"{r['metadata']['source_title']} § {r['metadata']['section_title']}")
            print(f'        {preview}...')
    return results


# Sanity check using the same failing query from before
_ = retrieve_bm25('Who won the 1st match of IPL 2026?', k=5, verbose=True)

Loaded 145 chunks for BM25 indexing.

BM25 index built on 145 chunks.

Sample tokenization preview:
  Source: 2026 Royal Challengers Bengaluru season
  Text: Royal Challengers Bengaluru is a franchise cricket team based in Bengaluru, Karnataka, India, which has been playing in the Indian Premier League since its inau...
  Tokens (first 20): ['royal', 'challengers', 'bengaluru', 'is', 'a', 'franchise', 'cricket', 'team', 'based', 'in', 'bengaluru', 'karnataka', 'india', 'which', 'has', 'been', 'playing', 'in', 'the', 'indian']

Query: 'Who won the 1st match of IPL 2026?'
  [1]  12.08  (cricbuzz_match )  IPL 2026 — 1st Match: RCB vs SRH § 1st Match
        In the 1st Match of the 2026 Indian Premier League (IPL 2026), played at M.Chinnaswamy Stadium, Bengaluru on 28 March 2026, ROYAL CHALLENGER...
  [2]  11.23  (wikipedia      )  2026 Chennai Super Kings season § Squad
        Players with international caps as of start of 2026 IPL are listed in bold. Ages are as of 26 March 2026 (2026-0

## Hybrid retrieval with Reciprocal Rank Fusion (RRF)


In [23]:
#Hybrid retrieval combining dense + BM25 via Reciprocal Rank Fusion.
#Each retriever returns its top k_dense / k_sparse candidates. RRF fuses them into a single ranking; we return the top `k` of the fused list.
#Returns the same shape as retrieve_dense and retrieve_bm25 so downstream code (rag_answer, evaluation) doesn't need to change.

RRF_K = 60

def retrieve_hybrid(
    query: str,
    k: int = 5,
    k_dense: int = 20,
    k_sparse: int = 20,
    verbose: bool = False,
) -> list[dict]:

    dense_results = retrieve_dense(query, k=k_dense)
    sparse_results = retrieve_bm25(query, k=k_sparse)

    # Build a {chunk_id -> rank} map for each retriever
    dense_ranks = {r['chunk_id']: i + 1 for i, r in enumerate(dense_results)}
    sparse_ranks = {r['chunk_id']: i + 1 for i, r in enumerate(sparse_results)}

    # Pool of all chunk_ids seen by either retriever
    all_ids = set(dense_ranks) | set(sparse_ranks)

    # Compute RRF score for each pooled chunk
    rrf_scores = {}
    for chunk_id in all_ids:
        score = 0.0
        if chunk_id in dense_ranks:
            score += 1.0 / (RRF_K + dense_ranks[chunk_id])
        if chunk_id in sparse_ranks:
            score += 1.0 / (RRF_K + sparse_ranks[chunk_id])
        rrf_scores[chunk_id] = score

    # We need the actual chunk data
    chunk_lookup = {r['chunk_id']: r for r in dense_results}
    for r in sparse_results:
        chunk_lookup.setdefault(r['chunk_id'], r)

    # Sort by RRF score, take top-k
    top_ids = sorted(rrf_scores, key=lambda cid: rrf_scores[cid], reverse=True)[:k]
    results = []
    for cid in top_ids:
        base = chunk_lookup[cid]
        results.append({
            'chunk_id': cid,
            'text': base['text'],
            'metadata': base['metadata'],
            'score': rrf_scores[cid],
            'rrf_components': {
                'dense_rank': dense_ranks.get(cid),
                'sparse_rank': sparse_ranks.get(cid),
            },
        })

    if verbose:
        print(f'Query: {query!r}\n')
        for i, r in enumerate(results, 1):
            src = r['metadata'].get('source_type', 'wikipedia')
            d = r['rrf_components']['dense_rank']
            s = r['rrf_components']['sparse_rank']
            d_str = f'd={d:2d}' if d else 'd=--'
            s_str = f's={s:2d}' if s else 's=--'
            print(f'  [{i}] rrf={r["score"]:.4f}  ({d_str}, {s_str})  ({src:15s})  '
                  f"{r['metadata']['source_title']} § {r['metadata']['section_title']}")
    return results


# Smoke test with the same failing query
_ = retrieve_hybrid('Who won the 1st match of IPL 2026?', k=5, verbose=True)
print()
_ = retrieve_hybrid('How did Gujarat Titans perform in IPL 2026?', k=5, verbose=True)
print()
_ = retrieve_hybrid('Were there any super-over matches in IPL 2026?', k=5, verbose=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'Who won the 1st match of IPL 2026?'

  [1] rrf=0.0315  (d= 6, s= 1)  (cricbuzz_match )  IPL 2026 — 1st Match: SRH vs RCB § 1st Match
  [2] rrf=0.0277  (d=17, s= 8)  (cricbuzz_match )  IPL 2026 — 24th Match: MI vs PBKS § 24th Match
  [3] rrf=0.0164  (d= 1, s=--)  (wikipedia      )  2026 Indian Premier League § Lead
  [4] rrf=0.0161  (d=--, s= 2)  (wikipedia      )  2026 Chennai Super Kings season § Squad
  [5] rrf=0.0161  (d= 2, s=--)  (wikipedia      )  2026 Sunrisers Hyderabad season § Pre-season



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'How did Gujarat Titans perform in IPL 2026?'

  [1] rrf=0.0320  (d= 2, s= 3)  (cricbuzz_match )  IPL 2026 — 52nd Match: GT vs RR § 52nd Match
  [2] rrf=0.0318  (d= 4, s= 2)  (cricbuzz_match )  IPL 2026 — 46th Match: PBKS vs GT § 46th Match
  [3] rrf=0.0313  (d= 1, s= 7)  (cricbuzz_match )  IPL 2026 — 66th Match: GT vs CSK § 66th Match
  [4] rrf=0.0310  (d= 3, s= 6)  (cricbuzz_match )  IPL 2026 — 37th Match: CSK vs GT § 37th Match
  [5] rrf=0.0307  (d=10, s= 1)  (cricbuzz_match )  IPL 2026 — 14th Match: GT vs DC § 14th Match



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'Were there any super-over matches in IPL 2026?'

  [1] rrf=0.0313  (d= 1, s= 7)  (wikipedia      )  2026 Chennai Super Kings season § Pre-season
  [2] rrf=0.0299  (d=10, s= 4)  (cricbuzz_match )  IPL 2026 — 38th Match: KKR vs LSG § 38th Match
  [3] rrf=0.0295  (d= 5, s=11)  (cricbuzz_match )  IPL 2026 — 33rd Match: CSK vs MI § 33rd Match
  [4] rrf=0.0287  (d= 3, s=18)  (cricbuzz_match )  IPL 2026 — 22nd Match: CSK vs KKR § 22nd Match
  [5] rrf=0.0286  (d= 8, s=12)  (cricbuzz_match )  IPL 2026 — 59th Match: CSK vs LSG § 59th Match


## Cross-encoder re-ranking

Bi-encoders and BM25 score query and document independently, so they can be fooled by lexical accidents — the diagnostic case being "super-over matches" pulling CSK's *Super Kings* "Pre-season" chunk to the top. A cross-encoder scores the (query, chunk) pair jointly via cross-attention, which catches these cases at the cost of ~20× latency per pair. We mitigate that by only re-ranking the top-K candidates from the hybrid retriever, not the full corpus.

In [24]:
from sentence_transformers import CrossEncoder

CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
_cross_encoder = None  # module-level cache so repeated calls don't reload


def get_cross_encoder() -> CrossEncoder:
    #Lazy-load the cross-encoder once per kernel session.
    global _cross_encoder
    if _cross_encoder is None:
        print(f'Loading cross-encoder: {CROSS_ENCODER_MODEL}')
        _cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, max_length=512)
    return _cross_encoder


def retrieve_hybrid_rerank(
    query: str,
    k: int = 5,
    k_candidates: int = 20,
    verbose: bool = False,
) -> list[dict]:
    """Hybrid RRF retrieval, then cross-encoder re-ranking.

    1. Pull k_candidates from retrieve_hybrid (RRF over dense + BM25).
    2. Score each (query, chunk_text) pair with the cross-encoder.
    3. Return the top-k by cross-encoder score.

    Result shape matches the other retrievers (chunk_id, text, metadata, score),
    with two extras for the report's evidence trail:
      - hybrid_rank: position in the pre-rerank top-K
      - cross_encoder_score: the raw CE logit for this (query, chunk) pair
    The existing rrf_components dict from retrieve_hybrid is preserved.
    Note: `score` is overwritten with the CE score so downstream code that
    sorts on `score` still works; the RRF score is still recoverable from
    rrf_components if you need it.
    """
    candidates = retrieve_hybrid(query, k=k_candidates)
    if not candidates:
        return []

    ce = get_cross_encoder()
    pairs = [(query, c['text']) for c in candidates]
    ce_scores = ce.predict(pairs, show_progress_bar=False)

    for hybrid_rank, (cand, ce_score) in enumerate(zip(candidates, ce_scores), start=1):
        cand['hybrid_rank'] = hybrid_rank
        cand['cross_encoder_score'] = float(ce_score)
        cand['score'] = float(ce_score)  # primary score is now the CE score

    reranked = sorted(candidates, key=lambda c: c['cross_encoder_score'], reverse=True)
    top = reranked[:k]

    if verbose:
        print(f'Query: {query!r}\n')
        for i, r in enumerate(top, 1):
            src = r['metadata'].get('source_type', 'wikipedia')
            d = r['rrf_components']['dense_rank']
            s = r['rrf_components']['sparse_rank']
            d_str = f'd={d:2d}' if d else 'd=--'
            s_str = f's={s:2d}' if s else 's=--'
            print(f'  [{i}] ce={r["cross_encoder_score"]:+7.3f}  '
                  f'hybrid={r["hybrid_rank"]:2d}  ({d_str}, {s_str})  ({src:15s})  '
                  f"{r['metadata']['source_title']} § {r['metadata']['section_title']}")
            preview = r['text'].replace('\n', ' ')[:140]
            print(f'        {preview}...')
    return top


# Diagnostic: the same three queries we tested earlier
print('Diagnostic 1: "Who won the 1st match of IPL 2026?"')
_ = retrieve_hybrid_rerank('Who won the 1st match of IPL 2026?', k=5, verbose=True)

print('Diagnostic 2: "How did Gujarat Titans perform in IPL 2026?"')
_ = retrieve_hybrid_rerank('How did Gujarat Titans perform in IPL 2026?', k=5, verbose=True)

print('Diagnostic 3: "Were there any super-over matches in IPL 2026?"')
_ = retrieve_hybrid_rerank('Were there any super-over matches in IPL 2026?', k=5, verbose=True)

Diagnostic 1: "Who won the 1st match of IPL 2026?"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

Query: 'Who won the 1st match of IPL 2026?'

  [1] ce= +9.431  hybrid= 1  (d= 6, s= 1)  (cricbuzz_match )  IPL 2026 — 1st Match: SRH vs RCB § 1st Match
        In the 1st Match of the 2026 Indian Premier League (IPL 2026), played at M.Chinnaswamy Stadium, Bengaluru on 28 March 2026, Sunrisers Hydera...
  [2] ce= +6.527  hybrid=19  (d=--, s=10)  (cricbuzz_match )  IPL 2026 — 6th Match: KKR vs SRH § 6th Match
        In the 6th Match of the 2026 Indian Premier League (IPL 2026), played at Eden Gardens, Kolkata on 02 April 2026, KOLKATA KNIGHT RIDERS (KKR)...
  [3] ce= +6.519  hybrid= 2  (d=17, s= 8)  (cricbuzz_match )  IPL 2026 — 24th Match: MI vs PBKS § 24th Match
        In the 24th Match of the 2026 Indian Premier League (IPL 2026), played at Wankhede Stadium, Mumbai on 16 April 2026, Mumbai Indians (MI) fac...
  [4] ce= +6.507  hybrid= 8  (d=--, s= 4)  (cricbuzz_match )  IPL 2026 — 69th Match: MI vs RR § 69th Match
        In the 69th Match of the 2026 Indian Premier League (IPL 2026

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'How did Gujarat Titans perform in IPL 2026?'

  [1] ce= +6.589  hybrid= 9  (d= 7, s=14)  (cricbuzz_match )  IPL 2026 — Final: GT vs RCB § Final
        In the Final of the 2026 Indian Premier League (IPL 2026), played at Narendra Modi Stadium, Ahmedabad on 31 May 2026, Gujarat Titans (GT) fa...
  [2] ce= +6.539  hybrid= 1  (d= 2, s= 3)  (cricbuzz_match )  IPL 2026 — 52nd Match: GT vs RR § 52nd Match
        In the 52nd Match of the 2026 Indian Premier League (IPL 2026), played at Sawai Mansingh Stadium, Jaipur on 09 May 2026, Gujarat Titans (GT)...
  [3] ce= +6.398  hybrid=11  (d= 9, s=16)  (cricbuzz_match )  IPL 2026 — 4th Match: GT vs PBKS § 4th Match
        In the 4th Match of the 2026 Indian Premier League (IPL 2026), played at Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur,...
  [4] ce= +6.388  hybrid= 3  (d= 1, s= 7)  (cricbuzz_match )  IPL 2026 — 66th Match: GT vs CSK § 66th Match
        In the 66th Match of the 2026 Indian Premier League (IPL 2026)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'Were there any super-over matches in IPL 2026?'

  [1] ce= +7.216  hybrid= 2  (d=10, s= 4)  (cricbuzz_match )  IPL 2026 — 38th Match: KKR vs LSG § 38th Match
        In the 38th Match of the 2026 Indian Premier League (IPL 2026), played at Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Luck...
  [2] ce= +6.563  hybrid= 8  (d=15, s=10)  (cricbuzz_match )  IPL 2026 — 53rd Match: LSG vs CSK § 53rd Match
        In the 53rd Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 10 May 2026, Lucknow Super Gian...
  [3] ce= +6.429  hybrid=20  (d=12, s=--)  (cricbuzz_match )  IPL 2026 — 63rd Match: CSK vs SRH § 63rd Match
        In the 63rd Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 18 May 2026, Chennai Super King...
  [4] ce= +6.374  hybrid= 7  (d= 9, s=13)  (cricbuzz_match )  IPL 2026 — 18th Match: CSK vs DC § 18th Match
        In the 18th Match of the 2026 Indian Premier

## Enhanced pipeline wired into "rag_answer"

Same prompt template, same model, same grounding rules with changed retrieval step . This is what makes the baseline-vs-enhanced comparison clean, any difference in answer quality is attributable to retrieval.

In [25]:
# Registry of retrievers keyed by pipeline_version.
RETRIEVERS = {
    'v0_baseline':      lambda q, k: retrieve_dense(q, k=k),
    'v1_hybrid_rerank': lambda q, k: retrieve_hybrid_rerank(q, k=k, k_candidates=20),
}


def rag_answer(
    query: str,
    k: int = 5,
    model: str = 'quality',
    pipeline_version: str = 'v0_baseline',
    verbose: bool = False,
) -> dict:
    """End-to-end RAG: retrieve, prompt, generate.

    Dispatch on pipeline_version selects the retriever. Everything downstream —
    prompt template, LLM call, returned bundle — is identical across pipelines,
    so any difference in answer quality is attributable to retrieval.

    Returns a dict with the answer, the retrieved chunks, and the full prompt
    sent to the LLM, enough to debug or evaluate any single query.
    """
    if pipeline_version not in RETRIEVERS:
        raise ValueError(
            f'Unknown pipeline_version: {pipeline_version!r}. '
            f'Known: {sorted(RETRIEVERS)}'
        )

    results = RETRIEVERS[pipeline_version](query, k)
    context = format_passages(results)
    prompt = GROUNDED_PROMPT_TEMPLATE.format(context=context, question=query)

    answer = llm_complete(
        query=query,
        prompt=prompt,
        pipeline_version=pipeline_version,
        model=model,
    )

    if verbose:
        print(f'\nPipeline: {pipeline_version}')
        for i, r in enumerate(results, 1):
            preview = r['text'].replace('\n', ' ')[:120]
            print(f'  [{i}] {r["metadata"]["source_title"]} § {r["metadata"]["section_title"]}')
            print(f'      {preview}...')

    return {
        'query': query,
        'answer': answer,
        'retrieved': results,
        'prompt': prompt,
        'pipeline_version': pipeline_version,
    }


# Smoke test: same query, both pipelines, side-by-side
test_query = 'Were there any super-over matches in IPL 2026?'


print('BASELINE (v0_baseline)')

baseline_out = rag_answer(test_query, pipeline_version='v0_baseline', verbose=True)
print('\nANSWER:')
print(baseline_out['answer'])

print('-' * 78)
print('ENHANCED (v1_hybrid_rerank)')
enhanced_out = rag_answer(test_query, pipeline_version='v1_hybrid_rerank', verbose=True)
print('\nANSWER:')
print(enhanced_out['answer'])

BASELINE (v0_baseline)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Pipeline: v0_baseline
  [1] 2026 Chennai Super Kings season § Pre-season
      The 2026 Indian Premier League will be the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20...
  [2] 2026 Indian Premier League § Lead
      The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, was the 19th edition of the Indian Pr...
  [3] IPL 2026 — 22nd Match: CSK vs KKR § 22nd Match
      In the 22nd Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 14 April 20...
  [4] 2026 Mumbai Indians season § Pre-season
      The 2026 Indian Premier League is the 19th edition of the Indian Premier League (IPL), a professional Twenty20 (T20) cri...
  [5] IPL 2026 — 33rd Match: CSK vs MI § 33rd Match
      In the 33rd Match of the 2026 Indian Premier League (IPL 2026), played at Wankhede Stadium, Mumbai on 23 April 2026, Che...

ANSWER:
The provided context does not answer this question.
------------------------

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Pipeline: v1_hybrid_rerank
  [1] IPL 2026 — 38th Match: KKR vs LSG § 38th Match
      In the 38th Match of the 2026 Indian Premier League (IPL 2026), played at Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...
  [2] IPL 2026 — 53rd Match: LSG vs CSK § 53rd Match
      In the 53rd Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 10 May 2026...
  [3] IPL 2026 — 63rd Match: CSK vs SRH § 63rd Match
      In the 63rd Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 18 May 2026...
  [4] IPL 2026 — 18th Match: CSK vs DC § 18th Match
      In the 18th Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 11 April 20...
  [5] IPL 2026 — 22nd Match: CSK vs KKR § 22nd Match
      In the 22nd Match of the 2026 Indian Premier League (IPL 2026), played at MA Chidambaram Stadium, Chennai on 14 April 20...

ANSWER:
Yes, there was at least one super-over match in I

##Test Cases


In [26]:
# 25 questions across 4 categories.
#should_refuse=True marks questions whose correct behaviour is to produce the grounded-prompt refusal sentence rather than an answer.
#gold_chunk_ids is the set of corpus chunks that legitimately support the answer collected manually for testing purposes

TEST_SET = [
    {
    'id': 'q01',
    'category': 'simple_fact',
    'question': 'Who won the 16th match of IPL 2026?',
    'gold_answer': 'Rajasthan Royals beat Royal Challengers Bengaluru by 6 wickets.',
    'gold_chunk_ids': ['cricbuzz_match::149768'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q02',
    'category': 'simple_fact',
    'question': 'Where was the 5th match of IPL 2026 played?',
    'gold_answer': 'The 5th match of IPL 2026 was played at Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow',
    'gold_chunk_ids': ['cricbuzz_match::149662'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q03',
    'category': 'simple_fact',
    'question': 'What date was  IPL 2026 announced?',
    'gold_answer': 'In February 2026, it was announced that the 2026 season would commence 2 days later on 28 March.',
    'gold_chunk_ids': ['2026 Indian Premier League::03::00'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q04',
    'category': 'simple_fact',
    'question': 'When was the first match of IPL 2026 played?',
    'gold_answer': 'The first match of IPL 2026 was played on 28 March 2026',
    'gold_chunk_ids': ['cricbuzz_match::149618'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q05',
    'category': 'simple_fact',
    'question': 'Who had the broadcasting rights for IPL 2026?',
    'gold_answer': 'JioStar\'s Star Sports and JioHotstar hold the broadcasting rights for IPL 2026',
    'gold_chunk_ids': ['2026 Indian Premier League::06::00'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q06',
    'category': 'simple_fact',
    'question': 'Who was the coach of RCB in IPL 2026?',
    'gold_answer': 'The coach of RCB in IPL 2026 was Andy Flower',
    'gold_chunk_ids': ['2026 Royal Challengers Bengaluru season::01::00'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q07',
    'category': 'simple_fact',
    'question': 'On what date and at which venue did the 27th match of IPL 2026 take place?',
    'gold_answer': 'The 27th match was played on 18 April 2026 at Rajiv Gandhi International Stadium, Hyderabad between Sunrisers Hyderabad (SRH) and Chennai Super Kings (CSK).',
    'gold_chunk_ids': ['cricbuzz_match::151818'],
    'should_refuse': False,
    'notes': 'Single Cricbuzz chunk. Tests basic match-result retrieval.',
},
{
    'id': 'q08',
    'category': 'edge_case',
    'question': 'Who won IPL 2027?',
    'gold_answer': 'Refusal expected-out of temporal scope. The corpus only covers IPL 2026.',
    'gold_chunk_ids': [],
    'should_refuse': True,
    'notes': 'Tests refusal on plausibly-phrased future question. The grounded prompt should produce the refusal sentence.',
},
{
    'id': 'q09',
    'category': 'edge_case',
    'question': 'When will IPL 2027 be played?',
    'gold_answer': 'Refusal expected-out of temporal scope. The corpus only covers IPL 2026.',
    'gold_chunk_ids': [],
    'should_refuse': True,
    'notes': 'Tests refusal on plausibly-phrased future question. The grounded prompt should produce the refusal sentence.',
},
{
    'id': 'q10',
    'category': 'edge_case',
    'question': 'What is the population of Bangalore?',
    'gold_answer': 'Refusal expected-out of domain. The corpus only covers IPL 2026.',
    'gold_chunk_ids': [],
    'should_refuse': True,
    'notes': 'Tests refusal on plausibly-phrased future question. The grounded prompt should produce the refusal sentence.',
},
{
    'id': 'q11',
    'category': 'edge_case',
    'question': 'How was the weather of Chennai?',
    'gold_answer': 'Refusal expected-out of domain. The corpus only covers IPL 2026.',
    'gold_chunk_ids': [],
    'should_refuse': True,
    'notes': 'Tests refusal on plausibly-phrased future question. The grounded prompt should produce the refusal sentence.',
},
{
    'id': 'q12',
    'category': 'edge_case',
    'question': 'Which match had the highest total score?',
    'gold_answer': '35th match of IPL 2026 held on 25 April between Delhi Capitals (DC) and Punjab Kings (PBKS) had the highest total score of 529.',
    'gold_chunk_ids': ['cricbuzz_match::151891'],
    'should_refuse': False,
    'notes': 'Not a refusal test. unretrievable aggregation',
},
{
    'id': 'q13',
    'category': 'edge_case',
    'question': 'How many sixes did Virat Kohli hit?',
    'gold_answer': 'Refusal expected-out of domain. The corpus does not cover player profiles.',
    'gold_chunk_ids': [],
    'should_refuse': True,
    'notes': 'Tests refusal on plausibly-phrased future question. The grounded prompt should produce the refusal sentence.',
},
{
    'id': 'q14',
    'category': 'cross_document',
    'question': 'How many matches did KKR win in the league stage of IPL 2026?',
    'gold_answer': 'KKR won 6 of their 14 league matches.',  # count by hand from data
    'gold_chunk_ids': [
        'cricbuzz_match::149629',
    'cricbuzz_match::149673',
    'cricbuzz_match::149732',
    'cricbuzz_match::149757',
    'cricbuzz_match::151763',
    'cricbuzz_match::151796',
    'cricbuzz_match::151829',
    'cricbuzz_match::151924',
    'cricbuzz_match::151998',
    'cricbuzz_match::152064',
    'cricbuzz_match::152130',
    'cricbuzz_match::152163',
    'cricbuzz_match::152218',
    'cricbuzz_match::152263',
    ],
    'should_refuse': False,
    'notes': 'Aggregation across 14 chunks. Top-5 retrieval will be incomplete — interesting failure mode.',
},
{
    'id': 'q15',
    'category': 'cross_document',
    'question': 'How many matches were tied?',
    'gold_answer': '1 match was tied.',
    'gold_chunk_ids': ['cricbuzz_match::151924'],
    'should_refuse': False,
    'notes': 'Single chunk answer (only one tied match: KKR vs LSG, match 38). Tests whether retrieval finds the tied-match chunk on a tie-specific query.',
},
{
    'id': 'q16',
    'category': 'cross_document',
    'question': 'How many total matches were played in IPL 2026?',
    'gold_answer': 'IPL 2026 had total 74 matches.',
    'gold_chunk_ids': ['2026 Indian Premier League::00::00'],
    'should_refuse': False,
    'notes': 'Single chunk answer (only one tied match: KKR vs LSG, match 38). Tests whether retrieval finds the tied-match chunk on a tie-specific query.',
},
{
    'id': 'q17',
    'category': 'cross_document',
    'question': 'Who was Sanju Samson traded for, and between which teams?',
    'gold_answer': 'Chennai Super Kings traded Ravindra Jadeja and Sam Curran to Rajasthan Royals in return for Sanju Samson.',
    'gold_chunk_ids': ['2026 Chennai Super Kings season::02::00', '2026 Rajasthan Royals season::00::00','2026 Rajasthan Royals season::02::00'],
    'should_refuse': False,
    'notes': 'Retreival through 3 chunks',
},
{
    'id': 'q18',
    'category': 'cross_document',
    'question': 'Which teams won the playoff matches in IPL 2026?',
    'gold_answer': 'RCB beat GT in Qualifier 1; RR beat SRH in the Eliminator; GT beat RR in Qualifier 2',
    'gold_chunk_ids': ['cricbuzz_match::155376', 'cricbuzz_match::155387','cricbuzz_match::155398'],
    'should_refuse': False,
    'notes': 'Retreival through 3 chunks',
},
{
    'id': 'q20',
    'category': 'cross_document',
    'question': 'How did Mumbai Indians perform in IPL 2026?',
    'gold_answer': 'Mumbai Indians won 4 of their 14 league matches and finished 9th and got eliminated from the league.',
    'gold_chunk_ids': ['2026 Mumbai Indians season::00::00',
    '2026 Mumbai Indians season::01::00',
    '2026 Mumbai Indians season::02::00',
    '2026 Mumbai Indians season::03::00',
    '2026 Mumbai Indians season::04::00',
    '2026 Mumbai Indians season::07::00',
    'cricbuzz_match::149629',
    'cricbuzz_match::149695',
    'cricbuzz_match::149743',
    'cricbuzz_match::149812',
    'cricbuzz_match::151785',
    'cricbuzz_match::151845',
    'cricbuzz_match::151878',
    'cricbuzz_match::151954',
    'cricbuzz_match::151987',
    'cricbuzz_match::152020',
    'cricbuzz_match::152097',
    'cricbuzz_match::152141',
    'cricbuzz_match::152218',
    'cricbuzz_match::152252',
],
    'should_refuse': False,
    'notes': 'Retreival through multiple chunks',
},
{
    'id': 'q21',
    'category': 'ambiguous',
    'question': 'What happened with the Royals?',
    'gold_answer': 'Rajasthan Royals appointed Riyan Parag as captain after trading Sanju Samson to CSK. They reached the qualifiers but lost to GT.',
    'gold_chunk_ids': [
        '2026 Rajasthan Royals season::00::00',
    '2026 Rajasthan Royals season::02::00',
    '2026 Rajasthan Royals season::03::00',
    '2026 Rajasthan Royals season::04::00',
    '2026 Rajasthan Royals season::05::00',
    '2026 Rajasthan Royals season::07::00',
    'cricbuzz_match::149640',  # 3rd Match: Rajasthan Royals won by 8 wkts.
    'cricbuzz_match::149699',  # 9th Match: Rajasthan Royals won by 6 runs.
    'cricbuzz_match::149743',  # 13th Match: Rajasthan Royals won by 27 runs  -  11 overs game due to rai
    'cricbuzz_match::149768',  # 16th Match: Rajasthan Royals won by 6 wkts.
    'cricbuzz_match::151752',  # 21st Match: Sunrisers Hyderabad won by 57 runs.
    'cricbuzz_match::151829',  # 28th Match: Kolkata Knight Riders won by 4 wkts.
    'cricbuzz_match::151867',  # 32nd Match: Rajasthan Royals won by 40 runs.
    'cricbuzz_match::151902',  # 36th Match: Sunrisers Hyderabad won by 5 wkts.
    'cricbuzz_match::151943',  # 40th Match: Rajasthan Royals won by 6 wkts.
    'cricbuzz_match::151976',  # 43rd Match: Delhi Capitals won by 7 wkts.
    'cricbuzz_match::152075',  # 52nd Match: Gujarat Titans won by 77 runs.
    'cricbuzz_match::152185',  # 62nd Match: Delhi Capitals won by 5 wkts.
    'cricbuzz_match::152207',  # 64th Match: Rajasthan Royals won by 7 wkts.
    'cricbuzz_match::152252',  # 69th Match: Rajasthan Royals won by 30 runs.
    'cricbuzz_match::155387',  # Eliminator: Rajasthan Royals won by 47 runs.
    'cricbuzz_match::155398',  # Qualifier 2: Gujarat Titans won by 7 wkts.
    ],
    'should_refuse': False,
    'notes': 'Short-name disambiguation ("Royals" → RR). Tests whether the model resolves the team name from context.',
},
{
    'id': 'q22',
    'category': 'ambiguous',
    'question': 'Who replaced Sanju Samson?',
    'gold_answer': 'Riyan Parag replaced Sanju Samson as captain of Rajasthan Royals for the 2026 season.',
    'gold_chunk_ids': ['2026 Rajasthan Royals season::02::00','2026 Rajasthan Royals season::00::00'],
    'should_refuse': False,
    'notes': 'Directional ambiguity — "replaced where?" Answer is captaincy at RR, not his playing role at CSK. Tests context-driven disambiguation.',
},
{
    'id': 'q23',
    'category': 'ambiguous',
    'question': 'Which team had Ricky Ponting as coach?',
    'gold_answer': 'Ricky Ponting was the coach of Punjab Kings in IPL 2026.',
    'gold_chunk_ids': ['2026 Punjab Kings season::00::00'],
    'should_refuse': False,
    'notes': 'Tests coach-name lookup. Answer is in PBKS Lead; retrieval must surface a single specific chunk from a thinly-covered article.',
},
{
    'id': 'q24',
    'category': 'ambiguous',
    'question': 'How did Bengaluru do?',
    'gold_answer': 'Royal Challengers Bengaluru won 9 of the 14 matches and won the qualifier round against GT.',
    'gold_chunk_ids': [
        '2026 Royal Challengers Bengaluru season::00::00',
    '2026 Royal Challengers Bengaluru season::01::00',
    '2026 Royal Challengers Bengaluru season::02::00',
    '2026 Royal Challengers Bengaluru season::03::00',
    '2026 Royal Challengers Bengaluru season::06::00',
    '2026 Royal Challengers Bengaluru season::17::00',
    'cricbuzz_match::149618',  # 1st Match: Royal Challengers Bengaluru won by 6 wkts.
    'cricbuzz_match::149721',  # 11th Match: Royal Challengers Bengaluru won by 43 runs.
    'cricbuzz_match::149768',  # 16th Match: Rajasthan Royals won by 6 wkts.
    'cricbuzz_match::149812',  # 20th Match: Royal Challengers Bengaluru won by 18 runs.
    'cricbuzz_match::151774',  # 23rd Match: Royal Challengers Bengaluru won by 5 wkts.
    'cricbuzz_match::151807',  # 26th Match: Delhi Capitals won by 6 wkts.
    'cricbuzz_match::151889',  # 34th Match: Royal Challengers Bengaluru won by 5 wkts.
    'cricbuzz_match::151935',  # 39th Match: Royal Challengers Bengaluru won by 9 wkts.
    'cricbuzz_match::151965',  # 42nd Match: Gujarat Titans won by 4 wkts.
    'cricbuzz_match::152053',  # 50th Match: LSG won by 9 runs (19 Overs game due to rain, DLS Target 213
    'cricbuzz_match::152097',  # 54th Match: Royal Challengers Bengaluru won by 2 wkts.
    'cricbuzz_match::152130',  # 57th Match: Royal Challengers Bengaluru won by 6 wkts.
    'cricbuzz_match::152174',  # 61st Match: Royal Challengers Bengaluru won by 23 runs.
    'cricbuzz_match::152240',  # 67th Match: Sunrisers Hyderabad won by 55 runs.
    'cricbuzz_match::155376',  # Qualifier 1: Royal Challengers Bengaluru won by 92 runs.

    ],
    'should_refuse': False,
    'notes': 'Short-name disambiguation ("Bengaluru" → RCB; corpus contains only one Bengaluru team). Open-ended aggregation, similar profile to MI performance question.',
},
{
    'id': 'q25',
    'category': 'ambiguous',
    'question': 'Did Hyderabad win the opening match?',
    'gold_answer': 'No. Sunrisers Hyderabad lost the opening match (1st Match) of IPL 2026 to Royal Challengers Bengaluru by 6 wickets.',
    'gold_chunk_ids': ['cricbuzz_match::149618'],
    'should_refuse': False,
    'notes': 'Short-name + yes/no ambiguity ("Hyderabad" → SRH). Tests whether the model correctly answers "no" rather than affirming based on surface match of "Hyderabad" tokens.',
},
{
    'id': 'q26',
    'category': 'ambiguous',
    'question': 'How did Chennai do in IPL 2026?',
    'gold_answer': 'Chennai Super Kings won 6 out of 14 matches finishing at 8th position.',
    'gold_chunk_ids': [
      '2026 Chennai Super Kings season::00::00',
    '2026 Chennai Super Kings season::01::00',
    '2026 Chennai Super Kings season::02::00',
    '2026 Chennai Super Kings season::03::00',
    '2026 Chennai Super Kings season::04::00',
    '2026 Chennai Super Kings season::05::00',
    '2026 Chennai Super Kings season::08::00',
    'cricbuzz_match::149640',  # 3rd Match: Rajasthan Royals won by 8 wkts.
    'cricbuzz_match::149684',  # 7th Match: Punjab Kings won by 5 wkts.
    'cricbuzz_match::149721',  # 11th Match: Royal Challengers Bengaluru won by 43 runs.
    'cricbuzz_match::149790',  # 18th Match: Chennai Super Kings won by 23 runs.
    'cricbuzz_match::151763',  # 22nd Match: Chennai Super Kings won by 32 runs.
    'cricbuzz_match::151818',  # 27th Match: Sunrisers Hyderabad won by 10 runs.
    'cricbuzz_match::151878',  # 33rd Match: Chennai Super Kings won by 103 runs.
    'cricbuzz_match::151913',  # 37th Match: Gujarat Titans won by 8 wkts.
    'cricbuzz_match::151987',  # 44th Match: Chennai Super Kings won by 8 wkts.
    'cricbuzz_match::152031',  # 48th Match: Chennai Super Kings won by 8 wkts.
    'cricbuzz_match::152086',  # 53rd Match: Chennai Super Kings won by 5 wkts.
    'cricbuzz_match::152152',  # 59th Match: Lucknow Super Giants won by 7 wkts.
    'cricbuzz_match::152196',  # 63rd Match: Sunrisers Hyderabad won by 5 wkts.
    'cricbuzz_match::152229',  # 66th Match: Gujarat Titans won by 89 runs.
  ],
    'should_refuse': False,
    'notes': 'Short-name disambiguation ("Chennai" → CSK). Open-ended aggregation. Parallels MI and RCB performance questions.',
},
]

# Sanity checks for common mistakes
import json
from collections import Counter

chunks_for_check = json.loads(CHUNKS_PATH.read_text())
all_chunk_ids = {c['chunk_id'] for c in chunks_for_check}

# Count + distribution
print(f'Total questions: {len(TEST_SET)}')
print('By category:', dict(Counter(q['category'] for q in TEST_SET)))
print(f'Refusal questions: {sum(q["should_refuse"] for q in TEST_SET)}')

# Unique IDs
ids = [q['id'] for q in TEST_SET]
assert len(ids) == len(set(ids)), f'Duplicate question IDs: {[i for i in ids if ids.count(i) > 1]}'

# Refusal questions should have empty gold_chunk_ids and vice versa
for q in TEST_SET:
    if q['should_refuse']:
        assert q['gold_chunk_ids'] == [], (
            f"{q['id']}: should_refuse=True but has gold_chunk_ids — these are inconsistent."
        )
    else:
        assert q['gold_chunk_ids'], (
            f"{q['id']}: should_refuse=False but gold_chunk_ids is empty — "
            f"this question can never score above zero on retrieval metrics."
        )

# Every gold_chunk_id must exist in the actual corpus
missing = []
for q in TEST_SET:
    for cid in q['gold_chunk_ids']:
        if cid not in all_chunk_ids:
            missing.append((q['id'], cid))
assert not missing, f'Gold chunk IDs not in corpus: {missing}'

print('\nAll sanity checks passed ✓')

Total questions: 25
By category: {'simple_fact': 7, 'edge_case': 6, 'cross_document': 6, 'ambiguous': 6}
Refusal questions: 5

All sanity checks passed ✓


In [27]:
# Every question runs once per pipeline (50 LLM calls total). The llm_complete cache means re-running this cell is near-instant after the first pass.
#Results are saved to disk for the metrics cell to consume independently.

from pathlib import Path
import json
from tqdm import tqdm

EVAL_DIR = PROJECT_DIR / 'results'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_RESULTS_PATH = EVAL_DIR / 'run_results.json'

PIPELINES = ['v0_baseline', 'v1_hybrid_rerank']


def run_pipeline_on_testset(test_set, pipeline_version, k=5):
    """Run rag_answer over the full test set. Returns a list of dicts with
    everything needed for later evaluation: question, answer, retrieved chunk
    metadata, and the question's gold fields."""
    results = []
    for q in tqdm(test_set, desc=f'  {pipeline_version}'):
        bundle = rag_answer(
            q['question'],
            k=k,
            model='quality',
            pipeline_version=pipeline_version,
            verbose=False,
        )
        results.append({
            'id': q['id'],
            'category': q['category'],
            'question': q['question'],
            'gold_answer': q['gold_answer'],
            'gold_chunk_ids': q['gold_chunk_ids'],
            'should_refuse': q['should_refuse'],
            'pipeline_version': pipeline_version,
            'answer': bundle['answer'],
            # Compact retrieval record: just IDs + ranks, full text recoverable from chunks.json if needed during failure analysis
            'retrieved_ids': [r['chunk_id'] for r in bundle['retrieved']],
            'retrieved_texts': [r['text'] for r in bundle['retrieved']],
        })
    return results


all_results = {}
for pv in PIPELINES:
    print(f'\nRunning {pv}...')
    all_results[pv] = run_pipeline_on_testset(TEST_SET, pv)

RUN_RESULTS_PATH.write_text(json.dumps(all_results, ensure_ascii=False, indent=2))
print(f'\nSaved to {RUN_RESULTS_PATH.name}')
print(f'  baseline: {len(all_results["v0_baseline"])} results')
print(f'  enhanced: {len(all_results["v1_hybrid_rerank"])} results')


Running v0_baseline...


  v0_baseline:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  v0_baseline:   8%|▊         | 2/25 [00:00<00:02, 10.59it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  v1_hybrid_rerank:  96%|█████████▌| 24/25 [01:42<00:03,  3.40s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  v1_hybrid_rerank: 100%|██████████| 25/25 [01:44<00:00,  4.20s/it]



Saved to run_results.json
  baseline: 25 results
  enhanced: 25 results


In [28]:
# Checking three complementary metrics on the same retrieval output:
# Hit@5: did any gold chunk make it into the top-5? (binary, per query)
# Recall@5: what fraction of all gold chunks did we retrieve? (Low values here are diagnostic, not a bug.)
# MRR: reciprocal rank of the FIRST gold chunk to appear, 0 if none.
# Refusal questions (should_refuse=True) are excluded from retrieval metrics since they have no gold chunks. They are evaluated by refusal rate instead.

import pandas as pd
import json

all_results = json.loads(RUN_RESULTS_PATH.read_text())

REFUSAL_MARKER = 'context does not answer'  # substring from your refusal sentence


def compute_per_query_metrics(result):
    #Compute Hit@5, Recall@5, MRR, refused for one (query, pipeline) result
    gold = set(result['gold_chunk_ids'])
    retrieved = result['retrieved_ids']  # in rank order
    refused = REFUSAL_MARKER in result['answer'].lower()

    if not gold:
        # Refusal question (only the refusal flag matters)
        return {'hit_at_5': None, 'recall_at_5': None, 'mrr': None,
                'refused': refused}

    top5 = retrieved[:5]
    intersect = gold & set(top5)

    hit_at_5 = 1 if intersect else 0
    recall_at_5 = len(intersect) / len(gold)

    mrr = 0.0
    for rank, cid in enumerate(top5, start=1):
        if cid in gold:
            mrr = 1.0 / rank
            break

    return {'hit_at_5': hit_at_5, 'recall_at_5': recall_at_5, 'mrr': mrr,
            'refused': refused}


# Build a flat dataframe: one row per (question, pipeline)
rows = []
for pv, results in all_results.items():
    for r in results:
        m = compute_per_query_metrics(r)
        rows.append({
            'id': r['id'],
            'category': r['category'],
            'pipeline': pv,
            'should_refuse': r['should_refuse'],
            **m,
        })

df = pd.DataFrame(rows)

# Save per-query for the report's appendix and failure analysis
df.to_csv(EVAL_DIR / 'retrieval_metrics_per_query.csv', index=False)
print(f'Per-query metrics → retrieval_metrics_per_query.csv')

# Aggregate: by pipeline, by category, by (pipeline × category)
print('\n--- Retrieval metrics: by pipeline ---')
print(df[df['should_refuse'] == False]
      .groupby('pipeline')[['hit_at_5', 'recall_at_5', 'mrr']]
      .mean()
      .round(3))

print('\n--- Retrieval metrics: by pipeline × category of query ---')
print(df[df['should_refuse'] == False]
      .groupby(['pipeline', 'category'])[['hit_at_5', 'recall_at_5', 'mrr']]
      .mean()
      .round(3))

print('\n--- Refusal behaviour ---')
# For refusal questions, we check how often does each pipeline correctly refuse?
ref_df = df[df['should_refuse'] == True]
print(ref_df.groupby('pipeline')['refused'].mean().round(3))
# For non-refusal questions, we check how often does each pipeline incorrectly refuse?
non_ref_df = df[df['should_refuse'] == False]
print('\nFalse-refusal rate (refused when shouldn\'t have):')
print(non_ref_df.groupby('pipeline')['refused'].mean().round(3))

Per-query metrics → retrieval_metrics_per_query.csv

--- Retrieval metrics: by pipeline ---
                  hit_at_5  recall_at_5    mrr
pipeline                                      
v0_baseline           0.45        0.213  0.425
v1_hybrid_rerank      0.75        0.563  0.692

--- Retrieval metrics: by pipeline × category of query ---
                                 hit_at_5  recall_at_5    mrr
pipeline         category                                    
v0_baseline      ambiguous          0.833        0.367  0.833
                 cross_document     0.667        0.343  0.583
                 edge_case          0.000        0.000  0.000
                 simple_fact        0.000        0.000  0.000
v1_hybrid_rerank ambiguous          0.833        0.443  0.750
                 cross_document     0.833        0.601  0.833
                 edge_case          0.000        0.000  0.000
                 simple_fact        0.714        0.714  0.619

--- Refusal behaviour ---
pipeline
v0_b

In [29]:
#manual evaluation on 6 test cases

import json, pandas as pd

all_results = json.loads(RUN_RESULTS_PATH.read_text())

SAMPLE_IDS = ['q01', 'q07', 'q11']  # simple_fact, cross_document, ambiguous

print('Generation Quality Sample \n')
rows = []
for pid in ['v0_baseline', 'v1_hybrid_rerank']:
    for r in all_results[pid]:
        if r['id'] in SAMPLE_IDS:
            print(f"[{pid}] {r['id']} — {r['category']}")
            print(f"  Q:    {r['question']}")
            print(f"  Gold: {r['gold_answer'][:120]}")
            print(f"  Ans:  {r['answer'][:120]}")
            print()
            rows.append({
                'id': r['id'],
                'pipeline': pid,
                'category': r['category'],
                'question': r['question'],
                'gold_answer': r['gold_answer'],
                'answer': r['answer'],
            })

pd.DataFrame(rows).to_csv(EVAL_DIR / 'generation_sample.csv', index=False)
print('Saved to generation_sample.csv')


Generation Quality Sample 

[v0_baseline] q01 — simple_fact
  Q:    Who won the 16th match of IPL 2026?
  Gold: Rajasthan Royals beat Royal Challengers Bengaluru by 6 wickets.
  Ans:  The provided context does not answer this question.

[v0_baseline] q07 — simple_fact
  Q:    On what date and at which venue did the 27th match of IPL 2026 take place?
  Gold: The 27th match was played on 18 April 2026 at Rajiv Gandhi International Stadium, Hyderabad between Sunrisers Hyderabad 
  Ans:  The provided context does not answer this question.

[v0_baseline] q11 — edge_case
  Q:    How was the weather of Chennai?
  Gold: Refusal expected-out of domain. The corpus only covers IPL 2026.
  Ans:  The provided context does not answer this question.

[v1_hybrid_rerank] q01 — simple_fact
  Q:    Who won the 16th match of IPL 2026?
  Gold: Rajasthan Royals beat Royal Challengers Bengaluru by 6 wickets.
  Ans:  Rajasthan Royals won the 16th match of IPL 2026 [1].

[v1_hybrid_rerank] q07 — simple_fact
  